In [9]:
from dotenv import load_dotenv
import os
import openai 
from openai import OpenAI
from pprint import pprint

load_dotenv()
api_key = os.getenv("OPENAI_SERVICE_KEY")
openai.api_key = api_key

client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"

In [10]:
def basic_intelligence(prompt: str) -> str:
    responses = client.responses.create(model=MODEL, input=prompt)
    print(responses)
    return responses.output_text

basic_intelligence(prompt="What are AI agents?")


Response(id='resp_68b02eee5cac819582866f5f27cd716a0764a6cd46647b23', created_at=1756376814.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_68b02eeefb5881959838620cd3ec296b0764a6cd46647b23', content=[ResponseOutputText(annotations=[], text='AI agents are systems designed to perceive their environment, reason, and make decisions to achieve specific goals. They can be categorized into several types based on their capabilities and levels of intelligence. Here are some key features and types of AI agents:\n\n### Key Features\n\n1. **Autonomy**: AI agents can operate independently and make decisions without human intervention.\n  \n2. **Adaptability**: They can learn from their experiences and adjust their behavior based on new information.\n\n3. **Interactivity**: Many AI agents interact with users or other systems, often utilizing natural language processing.\n\n4. **Goal-Driven

'AI agents are systems designed to perceive their environment, reason, and make decisions to achieve specific goals. They can be categorized into several types based on their capabilities and levels of intelligence. Here are some key features and types of AI agents:\n\n### Key Features\n\n1. **Autonomy**: AI agents can operate independently and make decisions without human intervention.\n  \n2. **Adaptability**: They can learn from their experiences and adjust their behavior based on new information.\n\n3. **Interactivity**: Many AI agents interact with users or other systems, often utilizing natural language processing.\n\n4. **Goal-Driven**: They are designed to achieve specific objectives, optimizing their actions to reach those goals.\n\n### Types of AI Agents\n\n1. **Reactive Agents**: These agents operate based on current inputs without memory of past states. They respond directly to stimuli (e.g., simple rule-based systems).\n\n2. **Deliberative Agents**: These agents maintain a

In [13]:
def without_memory():
    response = client.responses.create(model=MODEL,
                                       input=[{
                                           "role": "user",
                                           "content": "Tell me a joke about programming",
                                       }])
    return response.output_text

def followup_with_memory(earlier_response: str):
    response = client.responses.create(model=MODEL,
                                       input=[{
                                           "role": "user",
                                           "content": "Tell me a joke about programming",
                                       },{
                                           "role": "assistant",
                                           "content": earlier_response,
                                       },{
                                           "role": "user",
                                           "content": "What did I ask earlier ?",
                                       }])
    return response.output_text
    
def followup_without_memory():
    response = client.responses.create(model=MODEL,
                                       input=[{
                                           "role": "user",
                                           "content": "What did I ask earlier ?",
                                       }])
    return response.output_text

joke_response = without_memory()
print(joke_response)

followup_response = followup_without_memory()
print(followup_response)

followup_response = followup_with_memory(joke_response)
print(followup_response)


Why do programmers prefer dark mode?

Because light attracts bugs!
I’m sorry, but I can’t recall past conversations. However, feel free to ask anything again, and I’ll do my best to help!
You asked for a joke about programming. Would you like to hear another one?


In [14]:
import json
import requests
from openai import OpenAI


def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]["temperature_2m"]


def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    raise ValueError(f"Unknown function: {name}")


def intelligence_with_tools(prompt: str) -> str:
    tools = [
        {
            "type": "function",
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        }
    ]

    input_messages = [{"role": "user", "content": prompt}]

    # Step 1: Call model with tools
    response = client.responses.create(
        model="gpt-4o",
        input=input_messages,
        tools=tools,
    )
    
    print(response)

    # Step 2: Handle function calls
    for tool_call in response.output:
        if tool_call.type == "function_call":
            # Step 3: Execute function
            name = tool_call.name
            args = json.loads(tool_call.arguments)
            result = call_function(name, args)

            # Step 4: Append function call and result to messages
            input_messages.append(tool_call)
            input_messages.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": str(result),
                }
            )

    # Step 5: Get final response with function results
    final_response = client.responses.create(
        model="gpt-4o",
        input=input_messages,
        tools=tools,
    )

    return final_response.output_text


result = intelligence_with_tools(prompt="What's the weather like in Bangalore today?")
print("Tool Calling Output:")
print(result)

Response(id='resp_68b0374bd4488194a3bf2796a55c126604a0e82642dffd73', created_at=1756378955.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-2024-08-06', object='response', output=[ResponseFunctionToolCall(arguments='{"latitude":12.9716,"longitude":77.5946}', call_id='call_MRJ4bdIiTnNFmkWFBa2DfAkL', name='get_weather', type='function_call', id='fc_68b0374c9d8481949be325ec07095ebd04a0e82642dffd73', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'latitude': {'type': 'number'}, 'longitude': {'type': 'number'}}, 'required': ['latitude', 'longitude'], 'additionalProperties': False}, strict=True, type='function', description='Get current temperature for provided coordinates in celsius.')], top_p=1.0, background=False, max_output_tokens=None, previous_response_id=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None